# Cirq integration

This notebook shows a simple example of how to use pyGSTi with Cirq. It has three sections:

1. Sets up pyGSTi.
2. Shows how pyGSTi circuits can be converted to Cirq circuits.
3. Shows how Cirq circuits can be converted into pyGSTi circuits.
4. Shows how the Cirq circuits can be run and the results loaded back into pyGSTi for analysis.

In [1]:
import cirq
import pygsti
from pygsti.modelpacks import smq1Q_XYI
from pygsti.circuits import Circuit
import numpy as np
import tqdm

## 1. Generate the GST circuits

### Make target gate set $\{R_{X}(\pi/2), R_{Y}(\pi/2),I\}$

In [2]:
target_model = smq1Q_XYI.target_model()

### Preparation and measurement fiducials, germs

In [3]:
preps = smq1Q_XYI.prep_fiducials()
effects = smq1Q_XYI.meas_fiducials()
germs = smq1Q_XYI.germs()

### Construct pyGSTi circuits

In [4]:
max_lengths = list(np.logspace(0, 10, 11, base=2, dtype=int))

In [5]:
print(max_lengths)

[np.int64(1), np.int64(2), np.int64(4), np.int64(8), np.int64(16), np.int64(32), np.int64(64), np.int64(128), np.int64(256), np.int64(512), np.int64(1024)]


In [6]:
pygsti_circuits = pygsti.circuits.gstcircuits.create_lsgst_circuits(target_model, preps, effects, germs, max_lengths)

In [7]:
len(pygsti_circuits)

1624

## 2. Convert to runable `cirq.Circuit`'s

### Setup

Now, we need to map the qubit names from pyGSTi (`0`, `1`, etc.) into cirq qubits. There's nothing special about `cirq.GridQubit(8, 3)`; it's just an example.

In [8]:
q0 = cirq.GridQubit(8, 3)
qubit_label_dict = {0: q0}

### Testing examples

Do an example conversion.

In [9]:
pygsti_circuit = pygsti_circuits[111]
print('pyGSTi:')
print(pygsti_circuit)
print('Cirq:')
print(pygsti_circuit.convert_to_cirq(qubit_label_dict))

pyGSTi:
Qubit 0 ---|Gxpi2|-|Gxpi2|-| |-| |-|Gxpi2|---

Cirq:
(8, 3): ───X^0.5───X^0.5───I───I───X^0.5───


Do another example conversion.

In [10]:
pygsti_circuit = pygsti_circuits[90]
print('pyGSTi:')
print(pygsti_circuit)
print('Cirq:')
print(pygsti_circuit.convert_to_cirq(qubit_label_dict))

pyGSTi:
Qubit 0 ---|Gypi2|-|Gypi2|-|Gypi2|-|Gypi2|-|Gxpi2|-|Gxpi2|-|Gxpi2|---

Cirq:
(8, 3): ───Y^0.5───Y^0.5───Y^0.5───Y^0.5───X^0.5───X^0.5───X^0.5───


Now, lets try the same thing but specifing a wait duration for the idle operation.

In [11]:
wait_duration = cirq.Duration(nanos=100)

In [12]:
pygsti_circuit = pygsti_circuits[111]
print('pyGSTi:')
print(pygsti_circuit)
print('Cirq:')
print(pygsti_circuit.convert_to_cirq(qubit_label_dict, wait_duration))

pyGSTi:
Qubit 0 ---|Gxpi2|-|Gxpi2|-| |-| |-|Gxpi2|---

Cirq:
(8, 3): ───X^0.5───X^0.5───WaitGate(100 ns)───WaitGate(100 ns)───X^0.5───


In [13]:
pygsti_circuit = pygsti_circuits[90]
print('pyGSTi:')
print(pygsti_circuit)
print('Cirq:')
print(pygsti_circuit.convert_to_cirq(qubit_label_dict, wait_duration))

pyGSTi:
Qubit 0 ---|Gypi2|-|Gypi2|-|Gypi2|-|Gypi2|-|Gxpi2|-|Gxpi2|-|Gxpi2|---

Cirq:
(8, 3): ───Y^0.5───Y^0.5───Y^0.5───Y^0.5───X^0.5───X^0.5───X^0.5───


### The real thing

Now, convert all the circuits.

In [14]:
cirq_circuits = [c.convert_to_cirq(qubit_label_dict, wait_duration) for c in tqdm.tqdm(pygsti_circuits)]

100%|██████████| 1624/1624 [00:00<00:00, 0.00it/s]


In [15]:
cirq_circuits

[,
 (8, 3): ───X^0.5───,
 (8, 3): ───Y^0.5───,
 (8, 3): ───X^0.5───X^0.5───,
 (8, 3): ───X^0.5───X^0.5───X^0.5───,
 (8, 3): ───Y^0.5───Y^0.5───Y^0.5───,
 (8, 3): ───X^0.5───Y^0.5───,
 (8, 3): ───X^0.5───X^0.5───X^0.5───X^0.5───,
 (8, 3): ───X^0.5───Y^0.5───Y^0.5───Y^0.5───,
 (8, 3): ───Y^0.5───X^0.5───,
 (8, 3): ───Y^0.5───Y^0.5───,
 (8, 3): ───Y^0.5───X^0.5───X^0.5───,
 (8, 3): ───Y^0.5───X^0.5───X^0.5───X^0.5───,
 (8, 3): ───Y^0.5───Y^0.5───Y^0.5───Y^0.5───,
 (8, 3): ───X^0.5───X^0.5───Y^0.5───,
 (8, 3): ───X^0.5───X^0.5───X^0.5───X^0.5───X^0.5───,
 (8, 3): ───X^0.5───X^0.5───Y^0.5───Y^0.5───Y^0.5───,
 (8, 3): ───X^0.5───X^0.5───X^0.5───Y^0.5───,
 (8, 3): ───X^0.5───X^0.5───X^0.5───X^0.5───X^0.5───X^0.5───,
 (8, 3): ───X^0.5───X^0.5───X^0.5───Y^0.5───Y^0.5───Y^0.5───,
 (8, 3): ───Y^0.5───Y^0.5───Y^0.5───X^0.5───,
 (8, 3): ───Y^0.5───Y^0.5───Y^0.5───X^0.5───X^0.5───,
 (8, 3): ───Y^0.5───Y^0.5───Y^0.5───X^0.5───X^0.5───X^0.5───,
 (8, 3): ───Y^0.5───Y^0.5───Y^0.5───Y^0.5───Y^0.5───Y^0.5

Note that we're missing the measurments and the first circuit is empty (it's should just be an idle). Otherwise, the results look good, and those things should be easy to fix.

## 3. Convert Cirq circuits to pyGSTi circuits
We also have support for converting a cirq circuit to a pyGSTi circuit, which is demonstrated below.
Begin by constructing a cirq circuit directly.

In [16]:
#create to cirq qubit objects
qubit_00 = cirq.GridQubit(0,0)
qubit_01 = cirq.GridQubit(0,1)
#define a series of Moment objects, which fill the same role as circuit layers in pyGSTi.
moment1 = cirq.Moment([cirq.XPowGate(exponent=.5).on(qubit_00), cirq.I(qubit_01)])
moment2 = cirq.Moment([cirq.I(qubit_00), cirq.I(qubit_01)])
#This weird looking gate is the so-called N gate.
moment3 = cirq.Moment([cirq.PhasedXZGate(axis_phase_exponent=0.14758361765043326, 
                                         x_exponent=0.4195693767448338, 
                                         z_exponent=-0.2951672353008665).on(qubit_00),
                    cirq.I(qubit_01)])
moment4 = cirq.Moment([cirq.H(qubit_00), (cirq.T**-1).on(qubit_01)])
moment5 = cirq.Moment([cirq.CNOT.on(qubit_00, qubit_01)])
cirq_circuit_example = cirq.Circuit([moment1, moment2, moment3, moment4, moment5])
print(cirq_circuit_example)

(0, 0): ───X^0.5───I───PhXZ(a=0.148,x=0.42,z=-0.295)───H──────@───
                                                              │
(0, 1): ───I───────I───I───────────────────────────────T^-1───X───


To convert this into a pyGSTi circuit we can use the `from_cirq` class method of the Circuit class.

In [17]:
converted_cirq_circuit_default = Circuit.from_cirq(cirq_circuit_example)
print(converted_cirq_circuit_default)

Qubit Q0_0 ---|Gxpi2|-| |-|Gn|-| Gh  |-|CQ0_1|---
Qubit Q0_1 ---|     |-| |-|  |-|Gtdag|-|TQ0_0|---



Above you can see the result of converting the circuit using the default conversion settings. The classmethod has multiple options for customizing the returned pyGSTi circuit.
1. By default the method constructs a mapping between cirq qubit objects and pygsti qubit labels based on the type of cirq qubit provided. E.g. a GridQubit gets mapped to `Q{row}_{col}` where row and col are the corresponding attribute values for the GridQubit. Something similar is done for NamedQubit and LineQubit objects. This can be overridden by passing in a dictionary for the `qubit_conversion` kwarg.

In [18]:
converted_cirq_circuit_custom_qubit_map = Circuit.from_cirq(cirq_circuit_example, qubit_conversion={qubit_00: 'Qalice', qubit_01: 'Qbob'})
print(converted_cirq_circuit_custom_qubit_map)

Qubit Qalice ---|Gxpi2|-| |-|Gn|-| Gh  |-| CQbob |---
Qubit Qbob   ---|     |-| |-|  |-|Gtdag|-|TQalice|---



2. By default cirq included idle gates explicitly on all qubits in a layer without a specified operation applied. In pygsti we typically treat these as implied, and so the default behavior is to strip these extra idles. This can be turned off by setting `remove_implied_idles` to `False`.

In [19]:
converted_cirq_circuit_implied_idles = Circuit.from_cirq(cirq_circuit_example, remove_implied_idles=True)
print(converted_cirq_circuit_implied_idles)

Qubit Q0_0 ---|Gxpi2|-| |-|Gn|-| Gh  |-|CQ0_1|---
Qubit Q0_1 ---|     |-| |-|  |-|Gtdag|-|TQ0_0|---



3. Layers consisting entirely of idle gates are by default converted to the default pyGSTi global idle convention or Label(()), or to a user specified replacement. This is controlled by the `global_idle_replacement_label` kwarg. The default value is the string 'auto', which will utilize the aforementioned default convention. Users can instead pass in either a string, which is converted to a corresponding Label object, or a circuit Label object directly. Finally, by passing in `None` the global idle replacement is not performed, and the full verbatim translation of that cirq layer is produced.

In [20]:
#auto is the default value, explicitly including here for comparison to alternative options.
converted_cirq_circuit_global_idle = Circuit.from_cirq(cirq_circuit_example, global_idle_replacement_label='auto')
print(converted_cirq_circuit_global_idle)

Qubit Q0_0 ---|Gxpi2|-| |-|Gn|-| Gh  |-|CQ0_1|---
Qubit Q0_1 ---|     |-| |-|  |-|Gtdag|-|TQ0_0|---



In [21]:
converted_cirq_circuit_global_idle_1 = Circuit.from_cirq(cirq_circuit_example, global_idle_replacement_label='Gbanana')
print(converted_cirq_circuit_global_idle_1)

Qubit Q0_0 ---|Gxpi2|-|Gbanana:Q0_0:Q0_1|-|Gn|-| Gh  |-|CQ0_1|---
Qubit Q0_1 ---|     |-|Gbanana:Q0_0:Q0_1|-|  |-|Gtdag|-|TQ0_0|---



In [22]:
from pygsti.baseobjs import Label
converted_cirq_circuit_global_idle_2 = Circuit.from_cirq(cirq_circuit_example, global_idle_replacement_label=Label('Gbanana', ('Q0_0','Q0_1')))
print(converted_cirq_circuit_global_idle_2)

Qubit Q0_0 ---|Gxpi2|-|Gbanana:Q0_0:Q0_1|-|Gn|-| Gh  |-|CQ0_1|---
Qubit Q0_1 ---|     |-|Gbanana:Q0_0:Q0_1|-|  |-|Gtdag|-|TQ0_0|---



In [23]:
converted_cirq_circuit_global_idle_3 = Circuit.from_cirq(cirq_circuit_example, global_idle_replacement_label= None)
print(converted_cirq_circuit_global_idle_3)

Qubit Q0_0 ---|Gxpi2|-|Gi|-|Gn|-| Gh  |-|CQ0_1|---
Qubit Q0_1 ---|     |-|Gi|-|  |-|Gtdag|-|TQ0_0|---



4. There is built-in support for converting _most_ Cirq gates into their corresponding built-in pyGSTi gate names (see `cirq_gatenames_standard_conversions` in `pygsti.tools.internalgates` for more on this). There is also a fallback behavior where if not found in the default map, the converter will search among the built-in gate unitaries for one that matches (up to a global phase). If this doesn't work for a particular gate of user interest, of you simply want to override the default mapping as needed, this can be done by passing in a custom dictionary for the `cirq_gate_conversion` kwarg.

In [24]:
custom_gate_map = pygsti.tools.internalgates.cirq_gatenames_standard_conversions()
custom_gate_map[cirq.H] = 'Gdefinitelynoth'
converted_cirq_circuit_custom_gate_map = Circuit.from_cirq(cirq_circuit_example, cirq_gate_conversion=custom_gate_map)
print(converted_cirq_circuit_custom_gate_map)

Qubit Q0_0 ---|Gxpi2|-| |-|Gn|-|Gdefinitelynoth|-|CQ0_1|---
Qubit Q0_1 ---|     |-| |-|  |-|     Gtdag     |-|TQ0_0|---



5. Cirq circuits containing arbitrary-angle parameterized rotations (`cirq.ZPowGate`, `cirq.XPowGate`, `cirq.YPowGate`, `cirq.CZPowGate`, and arbitrary `cirq.PhasedXZGate`s -- the shapes most releases of real Google hardware circuits are built from) also convert cleanly, even when there's no exact-match pyGSTi gate name for the specific angle. These map to the parameterized pyGSTi standard gates `Gzr`, `Gxr`, `Gyr`, `Gczr`, and `Gu3` respectively, with the angle(s) (in radians) carried as the resulting `Label`'s `args`. (`cirq.Rx`/`Ry`/`Rz` are subclasses of `X`/`Y`/`ZPowGate` in Cirq, so they're covered automatically.)

In [25]:
parameterized_cirq_circuit = cirq.Circuit([
    cirq.Moment([cirq.PhasedXZGate(axis_phase_exponent=0.15, x_exponent=0.42, z_exponent=-0.3).on(qubit_00)]),
    cirq.Moment([cirq.ZPowGate(exponent=0.61).on(qubit_00), cirq.YPowGate(exponent=-0.24).on(qubit_01)]),
    cirq.Moment([cirq.CZPowGate(exponent=0.37).on(qubit_00, qubit_01)]),
])
print(parameterized_cirq_circuit)
print(Circuit.from_cirq(parameterized_cirq_circuit))

(0, 0): ───PhXZ(a=0.15,x=0.42,z=-0.3)───Z^0.61────@────────
                                                  │
(0, 1): ────────────────────────────────Y^-0.24───@^0.37───
Qubit Q0_0 ---|Gu3(1.319468914507713,-2.0420352248333655,1.0995574287564276)|-|Gzr(1.9163715186897738) |-|Gczr;1.1623892818282235:Q0_0:Q0_1|---
Qubit Q0_1 ---|                                                             |-|Gyr(-0.7539822368615503)|-|Gczr;1.1623892818282235:Q0_0:Q0_1|---



6. Cirq circuits with Z-basis mid-circuit measurements (`cirq.measure`) also convert, each measured qubit becoming its own `'Iz'` instrument label (the same convention used by `convert_to_qiskit`); the measurement key itself is not preserved. By default, *terminal* measurements -- those not followed by any other operation on the same qubit(s) -- are dropped rather than converted, since a terminal Z-basis measurement is exactly pyGSTi's implicit end-of-circuit readout; set `drop_terminal_measurements=False` to keep them as explicit `'Iz'` labels instead. These lossy-conversion notices (a dropped terminal measurement, or a discarded measurement key) are emitted as `CirqInteropWarning`s; pass `lossy='ignore'` to `from_cirq` to suppress them, or `lossy='raise'` to promote them to a `ValueError`.

In [26]:
mcm_cirq_circuit = cirq.Circuit([
    cirq.Moment([cirq.X(qubit_00)]),
    cirq.Moment([cirq.measure(qubit_00, key='mid_circuit')]),
    cirq.Moment([cirq.Y(qubit_00)]),
    cirq.Moment([cirq.measure(qubit_00, key='terminal')]),
])
print(mcm_cirq_circuit)
#the mid-circuit measurement survives as 'Iz'; the terminal one is dropped by default.
print(Circuit.from_cirq(mcm_cirq_circuit))
#pass drop_terminal_measurements=False to keep the terminal measurement too.
print(Circuit.from_cirq(mcm_cirq_circuit, drop_terminal_measurements=False))

(0, 0): ───X───M('mid_circuit')───Y───M('terminal')───
Qubit Q0_0 ---|Gxpi|-|Iz|-|Gypi|---

Qubit Q0_0 ---|Gxpi|-|Iz|-|Gypi|-|Iz|---



/tmp/ipykernel_4826/2562051340.py:9: CirqInteropWarning: pyGSTi circuit mapping discards Cirq measurement keys.
  print(Circuit.from_cirq(mcm_cirq_circuit))
/tmp/ipykernel_4826/2562051340.py:9: CirqInteropWarning: Dropping terminal Z-basis measurement(s): these correspond to pyGSTi's implicit end-of-circuit readout. Pass drop_terminal_measurements=False to instead convert them to explicit 'Iz' instrument labels.
  print(Circuit.from_cirq(mcm_cirq_circuit))
/tmp/ipykernel_4826/2562051340.py:11: CirqInteropWarning: pyGSTi circuit mapping discards Cirq measurement keys.
  print(Circuit.from_cirq(mcm_cirq_circuit, drop_terminal_measurements=False))


Both `Gzr`/`Gxr`/`Gyr`/`Gczr`/`Gu3` and `'Iz'` labels round-trip back through `convert_to_cirq` as well.

In [27]:
print(Circuit.from_cirq(parameterized_cirq_circuit).convert_to_cirq({'Q0_0': qubit_00, 'Q0_1': qubit_01}))

(0, 0): ───PhXZ(a=0.15,x=0.42,z=-0.3)───Z^0.61────@────────
                                                  │
(0, 1): ───I────────────────────────────Y^-0.24───@^0.37───


## 4. Run the circuits

Add measurements to the circuits.

In [28]:
for circuit in cirq_circuits:
    circuit.append(cirq.measure(q0, key='result'))

Simulate the circuits (or run them on a real quantum computer!)

In [29]:
simulator = cirq.Simulator()
results = [simulator.run(circuit, repetitions=1000) for circuit in tqdm.tqdm(cirq_circuits)]

100%|██████████| 1624/1624 [00:00<00:00, 0.00it/s]


Load everything the results into a pyGSTi dataset.

In [30]:
dataset = pygsti.data.dataset.DataSet()
for pygsti_circuit, trial_result in zip(pygsti_circuits, results):
    dataset.add_cirq_trial_result(pygsti_circuit, trial_result, key='result')

Perform GST. `StandardGSTDesign` rebuilds the circuit list from the fiducials, germs and max lengths, pairs it with the dataset we just built, and the `StandardGST` protocol runs on the pair.

In [31]:
design = pygsti.protocols.StandardGSTDesign(target_model, preps, effects, germs, max_lengths)
data = pygsti.protocols.ProtocolData(design, dataset)
gst_results = pygsti.protocols.StandardGST(modes=["full TP", "Target"], target_model=target_model, verbosity=1).run(data)

-- Std Practice:  [##################################################] 100.0%  (Target) --


See what if finds.

In [32]:
mdl_estimate = gst_results.estimates['full TP'].models['stdgaugeopt']
print("2DeltaLogL(estimate, data): ", pygsti.tools.two_delta_logl(mdl_estimate, dataset))
print("2DeltaLogL(ideal, data): ", pygsti.tools.two_delta_logl(target_model, dataset))

2DeltaLogL(estimate, data):  73478.36141262301
2DeltaLogL(ideal, data):  1153.6475839363263
